In [1]:
!date

Fri Sep 18 20:08:57 PDT 2026


In [2]:
import pandas as pd
import numpy as np
import glob
import os
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

projdir = '/u/project/cluo/terencew/claude/project_ideas/pool_design'
sample = '20220928-IGVF-D0'
tree = 'ambisim'

In [3]:
### same join as 01a, plus the per-droplet coverage and call-type columns 01a dropped:
### NUM.SNPS / NUM.READS (how much information demuxlet actually had for this droplet),
### DROPLET.TYPE (SNG/DBL/AMB, not just the SNG boolean), and DIFF.LLK.SNG.DBL (the margin
### that drives the singlet-vs-doublet call, as opposed to DIFF.LLK.BEST.NEXT which drives
### the donor-identity call). Everything downstream in 01f needs these.
def load_pool_modality(job):
    pool, modality = job
    best_path = f'{projdir}/{tree}/{pool}/demux/demuxlet/{modality}/{sample}.best'
    if not os.path.exists(best_path):
        return None
    best = pd.read_csv(best_path, sep='\t')
    best['barcode'] = best['BARCODE'].str.replace('-1', '', regex=False)
    truth = pd.read_csv(f'{projdir}/{tree}/{pool}/{sample}/drop_data_rand.txt', sep='\t',
                        dtype={'sam': str, 'ct': str})
    truth = truth.rename(columns={'RNA_BC': 'barcode'})
    df = truth.merge(best[['barcode', 'NUM.SNPS', 'NUM.READS', 'DROPLET.TYPE',
                           'SNG.BEST.GUESS', 'DIFF.LLK.BEST.NEXT',
                           'SNG.BEST.LLK', 'SNG.NEXT.LLK', 'DIFF.LLK.SNG.DBL']],
                     on='barcode', how='inner')
    if modality == 'gex':
        df['ambient_frac'] = df['rna_nr_a'] / (df['rna_nr_a'] + df['rna_nr_c'])
        df['true_reads'] = df['rna_nr_a'] + df['rna_nr_c']
    else:
        df['ambient_frac'] = df['atac_nr_a'] / (df['atac_nr_a'] + df['atac_nr_c'])
        df['true_reads'] = df['atac_nr_a'] + df['atac_nr_c']
    df['called_donor'] = df['SNG.BEST.GUESS'].str.split(',').str[0]
    df['is_true_singlet'] = df['n'] == 1
    df['called_singlet'] = df['DROPLET.TYPE'] == 'SNG'
    df['correct'] = df['is_true_singlet'] & df['called_singlet'] & (df['called_donor'] == df['sam'])
    # donor identity right, regardless of how the SNG/DBL/AMB call went -- this is the part of
    # "accuracy" that donor genetic distance can actually act on
    df['donor_correct'] = df['is_true_singlet'] & (df['called_donor'] == df['sam'])
    df['ll_gap'] = df['DIFF.LLK.BEST.NEXT']
    df['sng_gap'] = df['SNG.BEST.LLK'] - df['SNG.NEXT.LLK']
    df['sng_dbl_gap'] = df['DIFF.LLK.SNG.DBL']
    df['droplet_type'] = df['DROPLET.TYPE']
    df['n_snps'] = df['NUM.SNPS']
    df['n_reads_demux'] = df['NUM.READS']
    universe, strategy, rep = pool.split('__')
    df['pool'] = pool
    df['universe'] = universe
    df['strategy'] = strategy
    df['rep'] = int(rep.replace('rep', ''))
    df['modality'] = modality
    df = df.rename(columns={'sam': 'true_donor'})
    return df[['pool', 'universe', 'strategy', 'rep', 'modality', 'barcode',
               'is_true_singlet', 'called_singlet', 'correct', 'donor_correct',
               'ambient_frac', 'll_gap', 'sng_gap', 'sng_dbl_gap', 'droplet_type',
               'n_snps', 'n_reads_demux', 'true_reads', 'called_donor', 'true_donor']]

In [4]:
### readiness against the full 132-pool design, both modalities per pool (CONVENTIONS rule 4)
all_pools = [l.strip() for l in open(f'{projdir}/{tree}/txt/pool_experiments.txt')]
best_files = glob.glob(f'{projdir}/{tree}/*/demux/demuxlet/*/{sample}.best')
have = {}
for p in best_files:
    pool = p.split(f'/{tree}/')[1].split('/')[0]
    modality = p.split('/demuxlet/')[1].split('/')[0]
    have.setdefault(pool, set()).add(modality)

ready = [p for p in all_pools if have.get(p, set()) == {'gex', 'atac'}]
partial = {p: sorted(have[p]) for p in all_pools if p in have and have[p] != {'gex', 'atac'}}
not_started = [p for p in all_pools if p not in have]
print(f'{len(ready)}/{len(all_pools)} pools ready (both modalities)')
print(f'{len(partial)} partial: {partial}')
print(f'{len(not_started)} not started: {not_started}')

128/132 pools ready (both modalities)
0 partial: {}
4 not started: ['AFR_only__greedy_maxkl__rep1', 'EUR_EAS__ancestry_balanced__rep3', 'all_3_major__ancestry_balanced__rep3', 'EUR_AFR__adversarial_mindist__rep1']


In [5]:
### which of the ready pools are new since the n=88 run that RESULTS.md is written from --
### these 40 are an untouched replication set for the distance-vs-accuracy null
analyzed88 = set(pd.read_csv(f'{projdir}/csv/ambisim/pool_summary.csv', sep='\t')['pool'])
heldout = sorted(set(ready) - analyzed88)
print(len(heldout), 'held-out pools')
pd.Series(heldout).str.split('__').str[0].value_counts()

40 held-out pools


all_5          7
SAS_only       6
AMR_only       5
EUR_AFR        5
AFR_only       4
all_3_major    4
EAS_only       4
EUR_only       3
EUR_EAS        2
Name: count, dtype: int64

In [6]:
jobs = [(pool, modality) for pool in ready for modality in ['gex', 'atac']]
with ProcessPoolExecutor(max_workers=10) as ex:
    results = list(tqdm(ex.map(load_pool_modality, jobs), total=len(jobs)))
scored = pd.concat([r for r in results if r is not None], ignore_index=True)

100%|██████████| 256/256 [00:39<00:00,  6.53it/s]


In [7]:
scored.head()

,pool,universe,strategy,rep,modality,barcode,is_true_singlet,called_singlet,correct,donor_correct,ambient_frac,ll_gap,sng_gap,sng_dbl_gap,droplet_type,n_snps,n_reads_demux,true_reads,called_donor,true_donor
0,AFR_only__greedy_maxmean__rep1,AFR_only,greedy_maxmean,1,gex,AAACAGCCAAACAACA,True,True,True,True,0.098320,17.15,83.09,17.15,SNG,3232,3391,22915,HG02678,HG02678
1,AFR_only__greedy_maxmean__rep1,AFR_only,greedy_maxmean,1,gex,AAACAGCCAAACATAG,True,True,True,True,0.156914,10.20,106.56,10.20,SNG,4130,4488,30010,HG02816,HG02816
2,AFR_only__greedy_maxmean__rep1,AFR_only,greedy_maxmean,1,gex,AAACAGCCAAACCCTA,True,True,True,True,0.391450,-0.23,47.29,-0.23,SNG,2582,2843,19977,NA19922,NA19922
3,AFR_only__greedy_maxmean__rep1,AFR_only,greedy_maxmean,1,gex,AAACAGCCAAACCTAT,True,True,True,True,0.314493,-1.42,9.73,-1.42,SNG,718,741,5237,NA20274,NA20274
4,AFR_only__greedy_maxmean__rep1,AFR_only,greedy_maxmean,1,gex,AAACAGCCAAACCTTG,True,True,True,True,0.184638,19.78,156.28,19.78,SNG,6682,7458,51490,HG03136,HG03136


In [8]:
scored.shape

(2303744, 20)

In [9]:
complete_pools = scored.groupby('pool')['modality'].nunique()
complete_pools = complete_pools[complete_pools == 2].index
print(len(complete_pools), 'pools with both modalities scored')
scored = scored[scored['pool'].isin(complete_pools)].reset_index(drop=True)
scored['heldout'] = ~scored['pool'].isin(analyzed88)

128 pools with both modalities scored


In [10]:
scored.shape

(2303744, 21)

In [11]:
### checkpoint. written next to droplet_scores.csv rather than over it -- 01b/01c/01d are
### still wired to the n=88 file and RESULTS.md quotes numbers from it
outdir = f'{projdir}/csv/ambisim'
scored.to_csv(f'{outdir}/droplet_scores_cov.csv', sep='\t', index=False)

In [12]:
!date

Fri Sep 18 20:10:03 PDT 2026
